# 08 · Prefix caching with a token trie (RadixAttention-style)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/08_prefix_cache.ipynb) &nbsp; 📖 Concept deck: [PagedAttention and prefix caching](https://nilesh-salpe.github.io/visual-ai/inference/05-paged-attention/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a cache that reuses the KV of shared prompt prefixes (system prompts, few-shot examples, multi-turn chat) with LRU eviction of leaves, as in SGLang's RadixAttention, simplified to one token per node.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

print('no extra setup')

## Task 1: PrefixCache

Store cached token sequences in a trie (one node per token):
- `match(tokens)`: return how many leading tokens are already cached, and mark those nodes as used now.
- `insert(tokens)`: add any missing nodes and mark the whole path as used now. Then, while `size() > capacity`, **evict the least-recently-used leaf** (break ties by lower creation order). Evicting a leaf can turn its parent into a new leaf.
- `size()`: number of cached tokens (nodes, excluding the root).

Use a counter that increments on every `match` or `insert` call as the clock.

In [ ]:
class PrefixCache:
    def __init__(self, capacity):
        # TODO
        raise NotImplementedError
    def match(self, tokens): raise NotImplementedError
    def insert(self, tokens): raise NotImplementedError
    def size(self): raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_PrefixCache(C):
    c = C(capacity=12)
    sys = [1, 2, 3, 4, 5]
    c.insert(sys + [10, 11]); c.insert(sys + [20, 21, 22])
    assert c.size() == 10, f"shared prefix stored once: expected 10 nodes, got {c.size()}"
    assert c.match(sys + [20, 21, 99]) == 7, "should match 5 shared + 2 tokens"
    assert c.match([9, 9]) == 0
    c.insert(sys + [30, 31, 32, 33])      # +4 -> 14 > 12: evict 2 LRU leaves from the [10, 11] branch
    assert c.size() == 12, f"capacity not enforced: {c.size()}"
    assert c.match(sys + [10, 11]) == 5, "the least recently used branch [10, 11] should have been evicted"
    assert c.match(sys + [20, 21, 22]) == 8, "recently used branch must survive"

check("PrefixCache", _test_PrefixCache, PrefixCache)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: PrefixCache
class PrefixCache_solution:
    class _N:
        __slots__ = ('kids', 'parent', 'tok', 'used', 'order')
        def __init__(s, parent, tok, order): s.kids, s.parent, s.tok, s.used, s.order = {}, parent, tok, 0, order
    def __init__(self, capacity):
        self.cap, self.clock, self.n, self.created = capacity, 0, 0, 0
        self.root = self._N(None, None, -1)
    def match(self, tokens):
        self.clock += 1; node, k = self.root, 0
        for t in tokens:
            if t not in node.kids: break
            node = node.kids[t]; node.used = self.clock; k += 1
        return k
    def insert(self, tokens):
        self.clock += 1; node = self.root
        for t in tokens:
            if t not in node.kids:
                self.created += 1; node.kids[t] = self._N(node, t, self.created); self.n += 1
            node = node.kids[t]; node.used = self.clock
        while self.n > self.cap:
            leaves, stack = [], [self.root]
            while stack:
                x = stack.pop()
                if x is not self.root and not x.kids: leaves.append(x)
                stack.extend(x.kids.values())
            v = min(leaves, key=lambda x: (x.used, x.order))
            del v.parent.kids[v.tok]; self.n -= 1
    def size(self): return self.n

check("PrefixCache (reference)", _test_PrefixCache, PrefixCache_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Multi-user chat: every request starts with the same 200-token system prompt.
r = np.random.default_rng(0); sysp = list(range(1000, 1200)); c = PrefixCache_solution(capacity=3000)
hit = total = 0
for _ in range(200):
    req = sysp + list(r.integers(0, 50, int(r.integers(20, 80))))
    hit += c.match(req); total += len(req); c.insert(req)
print(f"prefix-cache hit rate over 200 requests: {hit/total:.0%} of prompt tokens skipped prefill")

---
## Takeaways
- Requests often share prefixes (system prompts, few-shot examples, earlier chat turns). Caching their KV skips recomputing them in prefill.
- A tree keyed by tokens finds the longest cached prefix. LRU eviction of **leaves** keeps shared stems cached the longest.
- SGLang's RadixAttention (a compressed radix tree) reported up to **6.4×** throughput on structured LLM programs.